<a href="https://colab.research.google.com/github/Solo7602/web/blob/4lab/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV
from collections import defaultdict

# 1. Загрузка данных

url = 'https://grouplens.org/datasets/movielens/latest/ml-latest-small.zip'
import zipfile, io, requests
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

# 2. Базовые показатели

# 2.1 Среднее количество просмотров каждого фильма (здесь считаем количество оценок как просмотры)
views_per_movie = ratings.groupby('movieId').size().mean()
print(f"Среднее количество просмотров каждого фильма: {views_per_movie:.2f}")

# 2.2 Популярность жанров
# Жанры в movies разделены символом '|'
movies['genres_list'] = movies['genres'].str.split('|')

# Создаем DataFrame с жанрами и считаем популярность как сумму просмотров фильмов в жанре
merged = ratings.merge(movies[['movieId', 'genres_list']], on='movieId')
genres_exploded = merged.explode('genres_list')
genre_popularity = genres_exploded.groupby('genres_list').size().sort_values(ascending=False)
print("Популярность жанров:")
print(genre_popularity)

# 2.3 Топ-10 популярных фильмов по количеству оценок
top_10_movies = ratings.groupby('movieId').size().sort_values(ascending=False).head(10)
top_10_movies = top_10_movies.reset_index()
top_10_movies = top_10_movies.merge(movies[['movieId', 'title']], on='movieId')
print("Топ-10 популярных фильмов:")
print(top_10_movies[['title', 0]])

# 3. Графики

plt.figure(figsize=(14, 5))

# График популярности жанров
plt.subplot(1, 2, 1)
genre_popularity.plot(kind='bar')
plt.title('Популярность жанров (кол-во просмотров)')
plt.ylabel('Количество просмотров')
plt.xlabel('Жанр')
plt.xticks(rotation=45)

# График топ-10 популярных фильмов
plt.subplot(1, 2, 2)
plt.barh(top_10_movies['title'], top_10_movies[0])
plt.title('Топ-10 популярных фильмов (по количеству оценок)')
plt.xlabel('Количество оценок')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# 4. Система рекомендаций на основе контекста + матричная факторизация

# Используем библиотеку surprise для реализации SVD (матричная факторизация)

# Добавим простейший контекст: жанр входит в данные косвенно через id фильма (будем использовать ratings напрямую)

# Подготовка данных для surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(ratings, test_size=0.2, random_state=42)
trainset_suprise = Dataset.load_from_df(trainset[['userId', 'movieId', 'rating']], reader).build_full_trainset()

algo = SVD(n_factors=50, random_state=42)
algo.fit(trainset_suprise)

# 5. Оценка качества модели

# RMSE
testset_list = list(zip(testset['userId'], testset['movieId'], testset['rating']))
predictions = algo.test(testset_list)
rmse = accuracy.rmse(predictions, verbose=False)
print(f"RMSE модели: {rmse:.4f}")

# MAP@k — нужно реализовать вручную

def apk(actual, predicted, k=10):
    """
    Average Precision at k
    actual — set правильных фильмов (у которых рейтинг >= 4)
    predicted — список предсказанных фильмов
    """
    if len(actual) == 0:
        return 0.0
    score = 0.0
    hits = 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i+1)
    return score / min(len(actual), k)

def mapk(actual_list, predicted_list, k=10):
    return np.mean([apk(a, p, k) for a, p in zip(actual_list, predicted_list)])

# Формируем рекомендации для пользователей из теста

def get_top_n_recommendations(predictions, n=10):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = [iid for iid, _ in user_ratings[:n]]
    return top_n

# Получим топ-10 для каждого пользователя в тесте
top_n = get_top_n_recommendations(predictions, n=10)

# Настоящие положительные фильмы ( где рейтинг >=4 )
true_relevants = testset[testset['rating'] >= 4].groupby('userId')['movieId'].apply(set).to_dict()

actual_list = []
predicted_list = []

for user_id in top_n:
    actual = true_relevants.get(user_id, set())
    predicted = top_n[user_id]
    actual_list.append(actual)
    predicted_list.append(predicted)

map10 = mapk(actual_list, predicted_list, k=10)
print(f"MAP@10 модели: {map10:.4f}")

# 6. Настройка гиперпараметров с GridSearchCV surprise

param_grid = {'n_factors': [20, 50, 100],
              'lr_all': [0.002, 0.005],
              'reg_all': [0.02, 0.05]}
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

print(f"Лучший RMSE: {gs.best_score['rmse']:.4f}")
print(f"Лучшие параметры: {gs.best_params['rmse']}")

# Обучаем на лучших параметрах
best_algo = gs.best_estimator['rmse']
best_algo.fit(trainset_suprise)
predictions_best = best_algo.test(testset_list)
rmse_best = accuracy.rmse(predictions_best, verbose=False)
print(f"RMSE после настройки гиперпараметров: {rmse_best:.4f}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [6]:
!pip install -U numpy scikit-surprise
import importlib
importlib.reload(__import__('numpy'))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 53.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.5 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.